In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests

log = load_log()
print(f"Log loaded. Rows: {len(log)}")

Log loaded. Rows: 3


## Freedom House FIW Pipeline

**Source:** Freedom House Freedom in the World
**Access:** Direct download where publicly available; email data request may be required for latest edition
**Download instructions:** See `docs/instructions_data_maintenance.md` — FH_FIW section

### Framework usage
| Sub-component | Concept | Role |
|---------------|---------|------|
| Political Rights A (Electoral Process) | Electoral process | Primary tier 1 |
| Civil Liberties D (Expression and Belief) | Civil liberties + Media freedom | Primary tier 1 / tier 2 |
| Civil Liberties E (Associational Rights) | Civil society space | Primary tier 1 |
| Civil Liberties G (Personal Autonomy) | Civil liberties | Primary tier 1 |

In [6]:
import requests
import re
import io
from datetime import datetime

def get_fh_data():
    """
    Try to find and download the FH FIW all-data Excel file.
    Constructs candidate URLs from recent year/month combinations — no hardcoding needed.
    FH URL pattern: /sites/default/files/{YYYY}-{MM}/All_data_FIW_{START}-{END}.xlsx
    Publication months have historically been 02 or 03.
    """
    current_year = datetime.today().year
    candidate_urls = []
    
    # Generate candidate URLs for current and prior 2 years, months 01-04
    for year in range(current_year, current_year - 3, -1):
        for month in ['03', '02', '01', '04']:
            for end_year in range(year, year - 3, -1):
                url = f"https://freedomhouse.org/sites/default/files/{year}-{month}/All_data_FIW_2013-{end_year}.xlsx"
                candidate_urls.append(url)

    for url in candidate_urls:
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200 and 'text/html' not in response.headers.get('Content-Type', ''):
                print(f"Found: {url}")
                return url, response.content
        except Exception as e:
            continue
    
    return None, None

print("Searching for current FH FIW data file...")
FH_URL, FH_CONTENT = get_fh_data()

if FH_CONTENT:
    print(f"Size: {len(FH_CONTENT)/1024:.1f} KB")
    xl = pd.ExcelFile(io.BytesIO(FH_CONTENT), engine='openpyxl')
    data_sheet = xl.sheet_names[-1]
    print(f"Sheets: {xl.sheet_names}")
    print(f"Using data sheet: {data_sheet}")
else:
    print("\n⚠️ No public FH FIW data file found.")
    print("See docs/instructions_data_maintenance.md — FH_FIW section for manual download instructions.")

Searching for current FH FIW data file...
Found: https://freedomhouse.org/sites/default/files/2025-02/All_data_FIW_2013-2024.xlsx
Size: 488.0 KB
Sheets: ['Index', 'FIW13-25']
Using data sheet: FIW13-25


In [8]:
# Re-parse with correct headers — row 0 is the real header
df_raw = xl.parse(data_sheet, header=1)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

Shape: (2723, 44)
Columns: ['Country/Territory', 'Region', 'C/T', 'Edition', 'Status', 'PR rating', 'CL rating', 'A1', 'A2', 'A3', 'A', 'B1', 'B2', 'B3', 'B4', 'B', 'C1', 'C2', 'C3', 'C', 'Add Q', 'Add A', 'PR', 'D1', 'D2', 'D3', 'D4', 'D', 'E1', 'E2', 'E3', 'E', 'F1', 'F2', 'F3', 'F4', 'F', 'G1', 'G2', 'G3', 'G4', 'G', 'CL', 'Total']
  Country/Territory   Region C/T  Edition Status  PR rating  CL rating  A1  \
0          Abkhazia  Eurasia   t     2025     PF          5          5   2   
1       Afghanistan     Asia   c     2025     NF          7          7   0   
2           Albania   Europe   c     2025     PF          3          3   3   

   A2  A3  ...  F3  F4   F  G1  G2  G3  G4  G  CL  Total  
0   2   1  ...   1   1   4   1   1   2   1  5  22     39  
1   0   0  ...   0   0   0   0   1   0   1  2   5      6  
2   3   3  ...   2   3  10   3   2   2   2  9  40     68  

[3 rows x 44 columns]


In [9]:
# Columns to keep — identifiers plus framework sub-categories
# A = Electoral Process, D = Expression/Belief, E = Associational Rights, G = Personal Autonomy
KEEP_COLS = [
    'Country/Territory', 'C/T', 'Edition',
    'A1', 'A2', 'A3', 'A',          # Electoral Process sub-questions + total
    'D1', 'D2', 'D3', 'D4', 'D',    # Expression and Belief sub-questions + total
    'E1', 'E2', 'E3', 'E',          # Associational Rights sub-questions + total
    'G1', 'G2', 'G3', 'G4', 'G',    # Personal Autonomy sub-questions + total
]

fh = df_raw[KEEP_COLS].copy()

# Rename columns
fh = fh.rename(columns={
    'Country/Territory': 'country_name',
    'C/T':               'country_or_territory',
    'Edition':           'year',
    'A1': 'fh_a1', 'A2': 'fh_a2', 'A3': 'fh_a3', 'A': 'fh_a_electoral_process',
    'D1': 'fh_d1', 'D2': 'fh_d2', 'D3': 'fh_d3', 'D4': 'fh_d4', 'D': 'fh_d_expression_belief',
    'E1': 'fh_e1', 'E2': 'fh_e2', 'E3': 'fh_e3', 'E': 'fh_e_associational_rights',
    'G1': 'fh_g1', 'G2': 'fh_g2', 'G3': 'fh_g3', 'G4': 'fh_g4', 'G': 'fh_g_personal_autonomy',
})

# Keep countries only (c), drop territories (t)
fh = fh[fh['country_or_territory'] == 'c'].drop(columns=['country_or_territory'])

# Ensure year is integer
fh['year'] = fh['year'].astype(int)

# Sort
fh = fh.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {fh.shape}")
print(f"Years: {sorted(fh['year'].unique())}")
print(f"Countries: {fh['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (fh.isnull().sum() / len(fh) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(fh.head())

Shape: (2535, 20)
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Countries: 195

Missing values (%):
Series([], dtype: float64)
  country_name  year  fh_a1  fh_a2  fh_a3  fh_a_electoral_process  fh_d1  \
0  Afghanistan  2013      1      1      1                       3      2   
1  Afghanistan  2014      1      1      1                       3      2   
2  Afghanistan  2015      1      1      1                       3      1   
3  Afghanistan  2016      1      0      1                       2      2   
4  Afghanistan  2017      1      0      1                       2      2   

   fh_d2  fh_d3  fh_d4  fh_d_expression_belief  fh_e1  fh_e2  fh_e3  \
0      1      1      2                       6      2      2      1   
1      1      1      2                       6      2      1      1   
2      1      1      2           

In [10]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "fh_fiw_clean.csv")
fh.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {fh.shape}")

# Derive metadata from data — no hardcoding
latest_year = str(int(fh['year'].max()))
data_as_of_date = latest_year

# Update download log
update_entry(
    "FH_FIW",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="fh_fiw_clean.csv",
    latest_available_version=latest_year,
    notes=f"Sub-categories A, D, E, G plus sub-questions. Countries only (territories excluded). Auto-detected URL: {FH_URL}"
)

print_entry("FH_FIW")

Written: C:\Users\mjbou\governance-framework\data\processed\fh_fiw_clean.csv
Shape: (2535, 20)
[download_log] Updated entry for FH_FIW
  source_id: FH_FIW
  last_attempted_date: 2026-05-21
  last_successful_download_date: 2026-05-21
  data_as_of_date: 2025
  local_filename: fh_fiw_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: Sub-categories A, D, E, G plus sub-questions. Countries only (territories excluded). Auto-detected URL: https://freedomhouse.org/sites/default/files/2025-02/All_data_FIW_2013-2024.xlsx
